In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors/Normal_10_Stage_3.npy
/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors/Insomnia_11_Wake.npy
/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors/Normal_08_Wake.npy
/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors/Normal_08_Stage_2.npy
/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors/Insomnia_06_Stage_2.npy
/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors/Insomnia_05_Stage_2.npy
/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors/Insomnia_08_Stage_2.npy
/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors/Insomnia_06_Wake.npy
/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors/Insomnia_06_R

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE-CELL PIPELINE: Combined Feature Extraction + Dataset Build
# Features: Band Power + PAC + wSMI + Hjorth + LZ + Sharma Wavelet Norms
# Total: 132 (Script-1) + 108 (Script-2) = 240 features per epoch
# Paste this entire block into ONE Kaggle notebook cell and run.
# ═══════════════════════════════════════════════════════════════════════════════

# ── 0. DEPENDENCIES ────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "antropy",    "-q"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "PyWavelets", "-q"])

# ── 1. IMPORTS ─────────────────────────────────────────────────────────────────
from __future__ import annotations
import math
import json
import warnings
import numpy as np
import pywt
from pathlib import Path
from collections import defaultdict
from scipy.signal import butter, filtfilt, hilbert
from typing import Tuple

# ══════════════════════════════════════════════════════════════════════════════
# SECTION A — CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
SFREQ: float     = 256.0
EPOCH_SEC: float = 30.0
N_SAMPLES: int   = int(SFREQ * EPOCH_SEC)   # 7680

# 17-channel layout from signal_processing.py:
# idx: 0=F3  1=F4  2=C3  3=C4  4=O1  5=O2  6=A1  7=A2
#      8=C4A1 9=C3A2 10=F3A2 11=F4A1 12=O1A2 13=O2A1
#      14=EMG 15=EMG1 16=EMG2
EEG_DIFF_INDICES: list[int] = [8, 9, 10, 11, 12, 13]   # differential only

BANDS: dict[str, Tuple[float, float]] = {
    "delta": (0.5,  4.0),
    "theta": (4.0,  8.0),
    "alpha": (8.0,  13.0),
    "sigma": (12.0, 15.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 45.0),
}

STAGE_MAP: dict[str, int] = {
    "Wake": 0, "Stage_1": 1, "Stage_2": 2, "Stage_3": 3, "REM": 4
}
GROUP_MAP: dict[str, int] = {
    "Normal": 0, "Insomnia": 1, "insomaniac": 1
}

# ── CONFIGURE YOUR PATHS HERE ──────────────────────────────────────────────────
DATA_DIR   = "/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors"
OUTPUT_DIR = "/kaggle/working/ml_ready_combined"
# ──────────────────────────────────────────────────────────────────────────────


# ══════════════════════════════════════════════════════════════════════════════
# SECTION B — SHARED VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

def _check_epoch(epoch: np.ndarray) -> None:
    if epoch.ndim != 2:
        raise ValueError(f"Expected 2D (channels, samples), got {epoch.shape}")
    if epoch.shape[1] != N_SAMPLES:
        raise ValueError(f"Expected {N_SAMPLES} samples, got {epoch.shape[1]}. "
                         f"Check SFREQ={SFREQ}Hz constant.")
    if np.any(np.isnan(epoch)):
        raise ValueError("NaN in epoch — artifact masking failed upstream.")
    if np.any(np.isinf(epoch)):
        raise ValueError("Inf in epoch.")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION C — SCRIPT-1 FEATURE EXTRACTORS (132 features)
# ══════════════════════════════════════════════════════════════════════════════

def _bandpass(signal: np.ndarray, low: float, high: float, fs: float) -> np.ndarray:
    nyq  = fs / 2.0
    high = min(high, nyq - 0.1)
    b, a = butter(4, [low / nyq, high / nyq], btype='band')
    return filtfilt(b, a, signal, axis=-1)


# ── C1: Band Power (72) ────────────────────────────────────────────────────────
def compute_band_power(epoch: np.ndarray) -> np.ndarray:
    """
    Absolute + relative band power via FFT periodogram.
    I/O: (17, 7680) → (72,)
    """
    _check_epoch(epoch)
    eeg   = epoch[EEG_DIFF_INDICES, :]          # (6, 7680)
    freqs = np.fft.rfftfreq(N_SAMPLES, d=1.0 / SFREQ)
    psd   = (np.abs(np.fft.rfft(eeg, axis=-1)) ** 2) / (SFREQ * N_SAMPLES)

    total = psd[:, (freqs >= 1.0) & (freqs <= 40.0)].sum(axis=-1)  # (6,)

    abs_p, rel_p = [], []
    for low, high in BANDS.values():
        bp = psd[:, (freqs >= low) & (freqs < high)].sum(axis=-1)
        abs_p.append(bp)
        rel_p.append(bp / np.where(total > 0, total, 1e-12))

    return np.concatenate([
        np.stack(abs_p, axis=0).T.flatten(),   # (36,)
        np.stack(rel_p, axis=0).T.flatten(),   # (36,)
    ]).astype(np.float32)                       # (72,)


# ── C2: PAC — SO-Spindle coupling (6) ─────────────────────────────────────────
def compute_pac_mvl(epoch: np.ndarray) -> np.ndarray:
    """
    Mean Vector Length PAC: SO phase (0.5-1.25Hz) × sigma amplitude (12-15Hz).
    Physiological basis: SO-spindle coupling disrupted in insomnia NREM.
    I/O: (17, 7680) → (6,)
    """
    _check_epoch(epoch)
    eeg       = epoch[EEG_DIFF_INDICES, :]
    so_phase  = np.angle(hilbert(_bandpass(eeg, 0.5,  1.25, SFREQ), axis=-1))
    sigma_amp = np.abs(  hilbert(_bandpass(eeg, 12.0, 15.0, SFREQ), axis=-1))
    mvl       = np.abs(np.mean(sigma_amp * np.exp(1j * so_phase), axis=-1))
    return mvl.astype(np.float32)   # (6,)


# ── C3: wSMI connectivity (30) ────────────────────────────────────────────────
def _ordinal_symbols(signal: np.ndarray, order: int) -> np.ndarray:
    """Bandt-Pompe ordinal pattern encoding via Lehmer/factorial-number system."""
    weights   = np.array([math.factorial(order - 1 - k) for k in range(order)], dtype=np.int32)
    n_symbols = len(signal) - (order - 1)
    symbols   = np.zeros(n_symbols, dtype=np.int32)
    for i in range(n_symbols):
        symbols[i] = int(np.argsort(signal[i:i + order]).dot(weights))
    return symbols


def compute_wsmi(epoch: np.ndarray, orders: Tuple[int, ...] = (3, 6)) -> np.ndarray:
    """
    Weighted Symbolic Mutual Information — all 15 EEG channel pairs.
    Volume-conduction robust. Validated for sleep staging (King et al., 2013).
    I/O: (17, 7680) → (30,)  [15 pairs × 2 orders]
    """
    _check_epoch(epoch)
    eeg   = epoch[EEG_DIFF_INDICES, :]
    n_ch  = eeg.shape[0]
    pairs = [(i, j) for i in range(n_ch) for j in range(i + 1, n_ch)]  # 15

    eeg_z = (eeg - eeg.mean(axis=-1, keepdims=True)) / (eeg.std(axis=-1, keepdims=True) + 1e-12)

    features = []
    for order in orders:
        n_states = math.factorial(order)
        sym      = [_ordinal_symbols(eeg_z[c], order) for c in range(n_ch)]
        n_sym    = len(sym[0])

        for i, j in pairs:
            joint = np.zeros((n_states, n_states), dtype=np.float64)
            np.add.at(joint, (sym[i], sym[j]), 1.0)
            joint /= n_sym

            pi    = joint.sum(axis=1, keepdims=True)
            pj    = joint.sum(axis=0, keepdims=True)
            outer = pi * pj

            mask = (joint > 1e-12) & (outer > 1e-12)
            mi   = np.sum(joint[mask] * np.log(joint[mask] / outer[mask]))
            features.append(mi)

    return np.array(features, dtype=np.float32)   # (30,)


# ── C4: Hjorth Parameters (18) ────────────────────────────────────────────────
def compute_hjorth(epoch: np.ndarray) -> np.ndarray:
    """
    Activity, Mobility, Complexity per differential EEG channel.
    I/O: (17, 7680) → (18,)
    """
    _check_epoch(epoch)
    eeg = epoch[EEG_DIFF_INDICES, :]
    d1  = np.diff(eeg, axis=-1)
    d2  = np.diff(d1,  axis=-1)

    v0  = np.var(eeg, axis=-1)
    v1  = np.var(d1,  axis=-1)
    v2  = np.var(d2,  axis=-1)

    act  = v0
    mob  = np.sqrt(v1 / (v0 + 1e-12))
    comp = np.sqrt(v2 / (v1 + 1e-12)) / (mob + 1e-12)

    return np.stack([act, mob, comp], axis=0).T.flatten().astype(np.float32)  # (18,)


# ── C5: Lempel-Ziv Complexity (6) ─────────────────────────────────────────────
def compute_lempel_ziv(epoch: np.ndarray) -> np.ndarray:
    """
    Normalized LZ complexity per channel (median binarization).
    Reduced in SWS, elevated in Wake/REM — strong stage discriminator.
    I/O: (17, 7680) → (6,)
    """
    try:
        import antropy as ant
    except ImportError:
        warnings.warn("antropy missing — LZ features zeroed. pip install antropy")
        return np.zeros(6, dtype=np.float32)

    _check_epoch(epoch)
    eeg = epoch[EEG_DIFF_INDICES, :]
    return np.array([
        ant.lziv_complexity(eeg[c] > np.median(eeg[c]), normalize=True)
        for c in range(eeg.shape[0])
    ], dtype=np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION D — SCRIPT-2 FEATURE EXTRACTOR (108 features — Sharma 2021)
# ══════════════════════════════════════════════════════════════════════════════

def compute_sharma_wavelet_norms(epoch: np.ndarray, wavelet: str = 'bior3.3') -> np.ndarray:
    """
    Sharma 2021 wavelet norm features.
    5-level 1D DWT (bior3.3) → 6 sub-bands [cA5, cD5, cD4, cD3, cD2, cD1].
    Per sub-band: L1, L2, L-inf norms.
    I/O: (17, 7680) → (108,)  [6 channels × 6 sub-bands × 3 norms]
    """
    _check_epoch(epoch)
    eeg = epoch[EEG_DIFF_INDICES, :]  # (6, 7680)

    features = []
    for ch in range(eeg.shape[0]):
        coeffs = pywt.wavedec(eeg[ch, :], wavelet, level=5)  # [cA5, cD5, cD4, cD3, cD2, cD1]
        for sub_band in coeffs:
            features.extend([
                np.sum(np.abs(sub_band)),           # L1
                np.linalg.norm(sub_band, ord=2),    # L2
                np.max(np.abs(sub_band)),            # L-inf
            ])

    return np.array(features, dtype=np.float32)     # (108,)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION E — FEATURE NAME REGISTRY (240 total)
# ══════════════════════════════════════════════════════════════════════════════

# Script-1 names (132)
_FEATURE_NAMES_S1: list[str] = (
    [f"bp_abs_{b}_ch{c}" for c in range(6) for b in BANDS]             +  # 36
    [f"bp_rel_{b}_ch{c}" for c in range(6) for b in BANDS]             +  # 36
    [f"pac_mvl_ch{c}"    for c in range(6)]                             +  # 6
    [f"wsmi_ord{o}_pair{i}_{j}"
     for o in [3, 6]
     for i in range(6) for j in range(i + 1, 6)]                       +  # 30
    [f"hjorth_{p}_ch{c}" for c in range(6) for p in ["act","mob","comp"]] +  # 18
    [f"lz_ch{c}"         for c in range(6)]                                # 6
)

# Script-2 names (108)
_SUB_BANDS = ["A5", "D5", "D4", "D3", "D2", "D1"]
_NORMS     = ["L1", "L2", "Linf"]
_FEATURE_NAMES_S2: list[str] = [
    f"wvl_{sb}_{norm}_ch{c}"
    for c in range(6)
    for sb in _SUB_BANDS
    for norm in _NORMS
]

# Combined registry
FEATURE_NAMES: list[str] = _FEATURE_NAMES_S1 + _FEATURE_NAMES_S2

assert len(_FEATURE_NAMES_S1) == 132, f"FATAL: Script-1 feature count = {len(_FEATURE_NAMES_S1)}, expected 132"
assert len(_FEATURE_NAMES_S2) == 108, f"FATAL: Script-2 feature count = {len(_FEATURE_NAMES_S2)}, expected 108"
assert len(FEATURE_NAMES)     == 240, f"FATAL: Combined feature count = {len(FEATURE_NAMES)}, expected 240"


# ══════════════════════════════════════════════════════════════════════════════
# SECTION F — MASTER EXTRACTOR
# ══════════════════════════════════════════════════════════════════════════════

def extract_features(epoch: np.ndarray) -> np.ndarray:
    """
    Combined feature extraction from both pipelines.

    I/O: (17, 7680) float32 → (240,) float32

    Feature layout:
        [  0: 71]  Band power — absolute + relative (72)
        [ 72: 77]  PAC MVL — SO-spindle coupling     (6)
        [ 78:107]  wSMI connectivity                 (30)
        [108:125]  Hjorth parameters                 (18)
        [126:131]  Lempel-Ziv complexity              (6)
        [132:239]  Sharma wavelet norms (bior3.3)    (108)

    Raises ValueError on NaN, Inf, or wrong input shape.
    """
    _check_epoch(epoch)
    return np.concatenate([
        compute_band_power(epoch),             # 72
        compute_pac_mvl(epoch),                # 6
        compute_wsmi(epoch),                   # 30
        compute_hjorth(epoch),                 # 18
        compute_lempel_ziv(epoch),             # 6
        compute_sharma_wavelet_norms(epoch),   # 108
    ]).astype(np.float32)                      # 240


# ══════════════════════════════════════════════════════════════════════════════
# SECTION G — FILENAME PARSER
# ══════════════════════════════════════════════════════════════════════════════

def parse_filename(fname: str) -> tuple | None:
    """
    Parses all three naming conventions in the dataset:
        Normal_01_REM.npy
        Insomnia_03_Stage_2.npy
        insomaniac_Subject_02_Wake.npy

    Returns (group, subject_num, stage) or None if unrecognised.
    """
    stem  = Path(fname).stem
    parts = stem.split('_')

    if parts[0] == 'insomaniac' and len(parts) >= 4 and parts[1] == 'Subject':
        group = 'insomaniac'
        try:    subj_num = int(parts[2])
        except: return None
        stage = '_'.join(parts[3:])

    elif parts[0] in ('Normal', 'Insomnia') and len(parts) >= 3:
        group = parts[0]
        try:    subj_num = int(parts[1])
        except: return None
        stage = '_'.join(parts[2:])

    else:
        return None

    if stage not in STAGE_MAP:
        print(f"  [SKIP] Unknown stage '{stage}' in {fname}")
        return None

    return group, subj_num, stage


# ══════════════════════════════════════════════════════════════════════════════
# SECTION H — PER-FILE LOADER + EXTRACTOR
# ══════════════════════════════════════════════════════════════════════════════

def load_and_extract(
    npy_path: str, group: str, subj_num: int, stage: str, subj_idx: int
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Loads one .npy tensor, extracts 240 features epoch-by-epoch.

    Expected input tensor shape: (n_epochs, 17, 7680)

    Returns
    -------
    X       : (n_valid, 240) float32
    y_stage : (n_valid,)     int32
    y_group : (n_valid,)     int32
    s_ids   : (n_valid,)     int32
    """
    tensor = np.load(npy_path)

    if tensor.ndim != 3 or tensor.shape[1] != 17 or tensor.shape[2] != N_SAMPLES:
        raise ValueError(
            f"Shape mismatch: {Path(npy_path).name} → expected (N, 17, {N_SAMPLES}), "
            f"got {tensor.shape}.  If sfreq ≠ 256 Hz update N_SAMPLES."
        )

    features, bad = [], 0
    for i in range(tensor.shape[0]):
        try:
            fv = extract_features(tensor[i])
            if np.any(np.isnan(fv)) or np.any(np.isinf(fv)):
                bad += 1
            else:
                features.append(fv)
        except Exception:
            bad += 1

    if bad:
        print(f"    [WARN] {bad}/{tensor.shape[0]} epochs rejected in {Path(npy_path).name}")

    if not features:
        e = np.empty((0, 240), np.float32)
        return e, np.empty(0, np.int32), np.empty(0, np.int32), np.empty(0, np.int32)

    X   = np.stack(features)
    y_s = np.full(len(features), STAGE_MAP[stage], dtype=np.int32)
    y_g = np.full(len(features), GROUP_MAP[group], dtype=np.int32)
    s_i = np.full(len(features), subj_idx,         dtype=np.int32)
    return X, y_s, y_g, s_i


# ══════════════════════════════════════════════════════════════════════════════
# SECTION I — MAIN BUILD FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def build_dataset(data_dir: str, output_dir: str) -> None:
    data_path = Path(data_dir)
    out_path  = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    npy_files = sorted(data_path.glob("*.npy"))
    if not npy_files:
        raise FileNotFoundError(f"No .npy files in {data_dir}\nCheck DATA_DIR path above.")

    # ── Build deterministic subject registry ─────────────────────────────────
    registry: dict[str, int] = {}
    parsed:   list            = []

    for f in npy_files:
        result = parse_filename(f.name)
        if result is None:
            continue
        group, subj_num, stage = result
        key = f"{group}_{subj_num:02d}"
        if key not in registry:
            registry[key] = len(registry)
        parsed.append((f, group, subj_num, stage, registry[key]))

    print(f"\n{'═'*60}")
    print(f"Subjects found : {len(registry)}")
    print(f"Stage files    : {len(parsed)}")
    print(f"Registry       : {list(registry.keys())}")
    print(f"{'═'*60}\n")

    # ── Extract features for every file ──────────────────────────────────────
    all_X, all_ys, all_yg, all_sid = [], [], [], []
    counts: dict[str, dict[str, int]] = defaultdict(dict)

    for npy_file, group, subj_num, stage, subj_idx in parsed:
        print(f"  Processing: {npy_file.name}", end=" ... ", flush=True)
        X, ys, yg, sid = load_and_extract(str(npy_file), group, subj_num, stage, subj_idx)
        if X.shape[0] == 0:
            print("SKIPPED (0 valid epochs)")
            continue
        all_X.append(X);   all_ys.append(ys)
        all_yg.append(yg); all_sid.append(sid)
        key = f"{group}_{subj_num:02d}"
        counts[key][stage] = int(X.shape[0])
        print(f"{X.shape[0]} epochs")

    # ── Assemble final arrays ─────────────────────────────────────────────────
    X_fin   = np.concatenate(all_X).astype(np.float32)
    ys_fin  = np.concatenate(all_ys).astype(np.int32)
    yg_fin  = np.concatenate(all_yg).astype(np.int32)
    sid_fin = np.concatenate(all_sid).astype(np.int32)

    # ── Hard validation ───────────────────────────────────────────────────────
    n = X_fin.shape[0]
    assert X_fin.shape[1]    == 240, f"Feature dim = {X_fin.shape[1]}, expected 240"
    assert ys_fin.shape[0]   == n,   "Row count mismatch: y_stage"
    assert yg_fin.shape[0]   == n,   "Row count mismatch: y_group"
    assert sid_fin.shape[0]  == n,   "Row count mismatch: subject_ids"
    assert np.isnan(X_fin).sum() == 0, \
        "NaN in final matrix — a feature extractor returned NaN silently"

    # ── Save ──────────────────────────────────────────────────────────────────
    np.save(out_path / "X.npy",           X_fin)
    np.save(out_path / "y_stage.npy",     ys_fin)
    np.save(out_path / "y_group.npy",     yg_fin)
    np.save(out_path / "subject_ids.npy", sid_fin)

    stage_dist = {str(k): int(v) for k, v in zip(*np.unique(ys_fin, return_counts=True))}
    group_dist = {str(k): int(v) for k, v in zip(*np.unique(yg_fin, return_counts=True))}

    metadata = {
        "n_epochs":    n,
        "n_features":  int(X_fin.shape[1]),
        "n_subjects":  len(registry),
        "feature_layout": {
            "band_power_abs_rel": "indices 0–71   (72 features)",
            "pac_mvl":            "indices 72–77  (6 features)",
            "wsmi":               "indices 78–107 (30 features)",
            "hjorth":             "indices 108–125 (18 features)",
            "lempel_ziv":         "indices 126–131 (6 features)",
            "sharma_wavelet":     "indices 132–239 (108 features)",
        },
        "feature_names":              FEATURE_NAMES,
        "subject_registry":           registry,
        "epoch_counts_per_subject":   {k: dict(v) for k, v in counts.items()},
        "stage_map":                  STAGE_MAP,
        "group_map":                  GROUP_MAP,
        "class_distribution_stage":   stage_dist,
        "class_distribution_group":   group_dist,
    }
    with open(out_path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    # ── Final report ──────────────────────────────────────────────────────────
    print(f"\n{'═'*60}")
    print(f"DATASET SAVED TO: {out_path.resolve()}")
    print(f"  X.npy            : {X_fin.shape}  (dtype={X_fin.dtype})")
    print(f"  y_stage.npy      : {ys_fin.shape}  unique={np.unique(ys_fin)}")
    print(f"  y_group.npy      : {yg_fin.shape}  unique={np.unique(yg_fin)}")
    print(f"  subject_ids.npy  : {sid_fin.shape}  unique subjects={len(np.unique(sid_fin))}")
    print(f"  Stage distribution : {stage_dist}")
    print(f"  Group distribution : {group_dist}")
    print(f"  NaN count          : {np.isnan(X_fin).sum()}   ← must be 0")
    print(f"  Feature layout:")
    print(f"    [  0: 71]  Band power abs+rel   (72)")
    print(f"    [ 72: 77]  PAC MVL              (6)")
    print(f"    [ 78:107]  wSMI                 (30)")
    print(f"    [108:125]  Hjorth               (18)")
    print(f"    [126:131]  Lempel-Ziv           (6)")
    print(f"    [132:239]  Sharma wavelet norms (108)")
    print(f"    {'─'*30}")
    print(f"    Total                           240")
    print(f"{'═'*60}")

    # ── Per-subject epoch count table ─────────────────────────────────────────
    print(f"\n{'─'*60}")
    print(f"{'Subject':<22} {'Wake':>6} {'N1':>6} {'N2':>6} {'N3':>6} {'REM':>6}  Group")
    print(f"{'─'*60}")
    stage_cols = ["Wake", "Stage_1", "Stage_2", "Stage_3", "REM"]
    for key, idx in sorted(registry.items(), key=lambda x: x[1]):
        grp   = "Insomnia" if GROUP_MAP.get(key.split('_')[0], 0) == 1 else "Normal"
        row   = counts.get(key, {})
        cells = [str(row.get(s, 0)).rjust(6) for s in stage_cols]
        print(f"  {key:<20} {'  '.join(cells)}  {grp}")
    print(f"{'─'*60}")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION J — SMOKE TEST
# ══════════════════════════════════════════════════════════════════════════════
print("Running smoke test on synthetic epoch...")
np.random.seed(42)
_dummy = np.random.randn(17, N_SAMPLES).astype(np.float32) * 50e-6

# Validate sub-extractor shapes individually before full merge
assert compute_band_power(_dummy).shape          == (72,),  "Band power shape mismatch"
assert compute_pac_mvl(_dummy).shape             == (6,),   "PAC shape mismatch"
assert compute_wsmi(_dummy).shape                == (30,),  "wSMI shape mismatch"
assert compute_hjorth(_dummy).shape              == (18,),  "Hjorth shape mismatch"
assert compute_lempel_ziv(_dummy).shape          == (6,),   "LZ shape mismatch"
assert compute_sharma_wavelet_norms(_dummy).shape == (108,), "Sharma wavelet shape mismatch"

_fv = extract_features(_dummy)
assert _fv.shape       == (240,), f"Expected (240,), got {_fv.shape}"
assert np.isnan(_fv).sum() == 0,  "NaN in smoke-test features"
assert np.isinf(_fv).sum() == 0,  "Inf in smoke-test features"
print(f"  Sub-extractor shapes : all PASSED")
print(f"  Combined shape       : {_fv.shape}  ✓")
print(f"  NaNs=0, Infs=0       : PASSED\n")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION K — RUN
# ══════════════════════════════════════════════════════════════════════════════
build_dataset(data_dir=DATA_DIR, output_dir=OUTPUT_DIR)

Running smoke test on synthetic epoch...
  Sub-extractor shapes : all PASSED
  Combined shape       : (240,)  ✓
  NaNs=0, Infs=0       : PASSED


════════════════════════════════════════════════════════════
Subjects found : 22
Stage files    : 107
Registry       : ['Insomnia_01', 'Insomnia_03', 'Insomnia_04', 'Insomnia_05', 'Insomnia_06', 'Insomnia_07', 'Insomnia_08', 'Insomnia_09', 'Insomnia_10', 'Insomnia_11', 'Normal_01', 'Normal_02', 'Normal_03', 'Normal_04', 'Normal_05', 'Normal_06', 'Normal_07', 'Normal_08', 'Normal_09', 'Normal_10', 'Normal_11', 'insomaniac_02']
════════════════════════════════════════════════════════════

  Processing: Insomnia_01_REM.npy ... 44 epochs
  Processing: Insomnia_01_Stage_1.npy ... 191 epochs
  Processing: Insomnia_01_Stage_2.npy ... 103 epochs
  Processing: Insomnia_01_Stage_3.npy ... 21 epochs
  Processing: Insomnia_01_Wake.npy ... 421 epochs
  Processing: Insomnia_03_REM.npy ... 14 epochs
  Processing: Insomnia_03_Stage_1.npy ... 376 epochs
  Pro

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE-CELL DATA INSPECTION + NORMALIZATION AUDIT (COMBINED 240 FEATURES)
# Covers: Band Power + PAC + wSMI + Hjorth + LZ (132) + Sharma Wavelet (108)
# Run this BEFORE train_loso.py — catch distribution problems now.
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import json
import warnings
from pathlib import Path
from scipy import stats as sp_stats

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False
    warnings.warn("matplotlib missing — skipping plots. pip install matplotlib")

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", font_scale=0.9)
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
ML_DIR     = "/kaggle/working/ml_ready_combined"
OUTPUT_DIR = "/kaggle/working/inspection_combined"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

STAGE_NAMES = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM"}
GROUP_NAMES = {0: "Normal", 1: "Insomnia"}

# Feature block boundaries — must match feature_extraction_combined.py layout
FEATURE_BLOCKS = {
    "Band Power (abs)":    (0,    36),   # 36
    "Band Power (rel)":    (36,   72),   # 36
    "PAC MVL":             (72,   78),   # 6
    "wSMI":                (78,  108),   # 30
    "Hjorth":             (108,  126),   # 18
    "Lempel-Ziv":         (126,  132),   # 6
    "Sharma Wavelet":     (132,  240),   # 108  [L1, L2, Linf × 6 sub-bands × 6 ch]
}

# Sub-blocks within Sharma block for per-norm analysis
SHARMA_NORM_BLOCKS = {
    "L1 Norms":   list(range(132, 240, 3)),         # every 3rd starting at 132
    "L2 Norms":   list(range(133, 240, 3)),
    "Linf Norms": list(range(134, 240, 3)),
}

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — LOAD
# ══════════════════════════════════════════════════════════════════════════════
print("Loading combined dataset...")
p           = Path(ML_DIR)
X           = np.load(p / "X.npy")
y_stage     = np.load(p / "y_stage.npy")
y_group     = np.load(p / "y_group.npy")
subject_ids = np.load(p / "subject_ids.npy")

with open(p / "metadata.json") as f:
    meta = json.load(f)

feature_names    = meta["feature_names"]
subject_registry = meta["subject_registry"]
id_to_name       = {v: k for k, v in subject_registry.items()}

N, D = X.shape
print(f"X shape         : {X.shape}   (N={N} epochs, D={D} features)")
print(f"Unique subjects : {len(np.unique(subject_ids))}")
print(f"Unique stages   : {np.unique(y_stage)}")
print(f"Unique groups   : {np.unique(y_group)}")

assert D == 240, f"Expected 240 combined features, got {D}. Wrong ML_DIR?"
assert len(feature_names) == 240, \
    f"metadata feature_names length = {len(feature_names)}, expected 240"

# Verify block boundaries sum to 240
total_block_feats = sum(hi - lo for lo, hi in FEATURE_BLOCKS.values())
assert total_block_feats == 240, \
    f"FEATURE_BLOCKS total = {total_block_feats}, expected 240 — edit boundaries above."

print(f"\nFeature layout confirmed:")
for block_name, (lo, hi) in FEATURE_BLOCKS.items():
    print(f"  [{lo:>3}:{hi:>3}]  {block_name}  ({hi-lo} features)")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — NaN / Inf / Zero-Variance AUDIT
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 2: Pathological Value Audit")
print(f"{'═'*60}")

nan_per_feat = np.isnan(X).sum(axis=0)
inf_per_feat = np.isinf(X).sum(axis=0)
var_per_feat = np.var(X, axis=0)

n_nan_feats = (nan_per_feat > 0).sum()
n_inf_feats = (inf_per_feat > 0).sum()
n_zero_var  = (var_per_feat < 1e-12).sum()

print(f"Features with any NaN         : {n_nan_feats}  (must be 0)")
print(f"Features with any Inf         : {n_inf_feats}  (must be 0)")
print(f"Features with zero variance   : {n_zero_var}  (useless for ML)")

if n_zero_var > 0:
    zv_names = [feature_names[i] for i in np.where(var_per_feat < 1e-12)[0]]
    print(f"  Zero-variance features: {zv_names}")

# Per-block zero-variance breakdown
print(f"\nZero-variance count per block:")
for block_name, (lo, hi) in FEATURE_BLOCKS.items():
    zv = (var_per_feat[lo:hi] < 1e-12).sum()
    print(f"  {block_name:<22}: {zv}/{hi-lo}")

# Per-subject variance check (catch between-subject confounds)
print(f"\nPer-subject variance check (catch subject-constant features):")
per_subj_var    = np.zeros((len(subject_registry), D))
for sid in np.unique(subject_ids):
    per_subj_var[sid] = np.var(X[subject_ids == sid], axis=0)
mean_within_var = per_subj_var.mean(axis=0)
suspicious      = np.where(mean_within_var < (var_per_feat * 0.001))[0]
print(f"  Features near-constant within subjects but varying across: {len(suspicious)}")
if 0 < len(suspicious) < 20:
    print(f"  Names: {[feature_names[i] for i in suspicious]}")
    print(f"  NOTE: These may be between-subject confounds, not EEG signal.")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — DISTRIBUTION SHAPE AUDIT
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 3: Distribution Shape — Normality Screening")
print(f"{'═'*60}")

rng        = np.random.default_rng(42)
sample_idx = rng.choice(N, size=min(2000, N), replace=False)
X_sample   = X[sample_idx]

non_normal_count = 0
skew_vals, kurt_vals = [], []

for i in range(D):
    col = X_sample[:, i]
    skew_vals.append(sp_stats.skew(col))
    kurt_vals.append(sp_stats.kurtosis(col))   # excess kurtosis
    _, p_val = sp_stats.normaltest(col)        # D'Agostino k²
    if p_val < 0.05:
        non_normal_count += 1

skew_arr = np.array(skew_vals)
kurt_arr = np.array(kurt_vals)

print(f"Non-normal features (D'Agostino k², α=0.05) : {non_normal_count}/{D}")
print(f"Skewness — mean={skew_arr.mean():.2f}, median={np.median(skew_arr):.2f}, "
      f"max_abs={np.abs(skew_arr).max():.2f}")
print(f"Kurtosis — mean={kurt_arr.mean():.2f}, median={np.median(kurt_arr):.2f}, "
      f"max_abs={np.abs(kurt_arr).max():.2f}")

print(f"\nPer feature-block skewness profile:")
for block_name, (lo, hi) in FEATURE_BLOCKS.items():
    blk_skew = np.abs(skew_arr[lo:hi])
    flag     = "→ LOG TRANSFORM RECOMMENDED" if blk_skew.mean() > 1.5 else "→ acceptable"
    print(f"  {block_name:<22}: mean|skew|={blk_skew.mean():.2f}, "
          f"max|skew|={blk_skew.max():.2f}  {flag}")

# Sharma sub-block skewness (norm-type granularity)
print(f"\nSharma sub-block skewness (by norm type):")
for norm_name, indices in SHARMA_NORM_BLOCKS.items():
    blk_skew = np.abs(skew_arr[indices])
    print(f"  {norm_name:<15}: mean|skew|={blk_skew.mean():.2f}, "
          f"max|skew|={blk_skew.max():.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — DYNAMIC RANGE AUDIT
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 4: Dynamic Range per Feature Block")
print(f"{'═'*60}")
print(f"{'Block':<24} {'Min':>12} {'Max':>12} {'Mean':>12} {'Std':>12}")
print(f"{'─'*74}")

for block_name, (lo, hi) in FEATURE_BLOCKS.items():
    blk = X[:, lo:hi]
    print(f"  {block_name:<22} {blk.min():>12.4e} {blk.max():>12.4e} "
          f"{blk.mean():>12.4e} {blk.std():>12.4e}")

# Sharma norm-type range breakdown
print(f"\nSharma sub-block dynamic range:")
print(f"{'Norm':<15} {'Min':>12} {'Max':>12} {'Mean':>12} {'Std':>12}")
print(f"{'─'*60}")
for norm_name, indices in SHARMA_NORM_BLOCKS.items():
    blk = X[:, indices]
    print(f"  {norm_name:<13} {blk.min():>12.4e} {blk.max():>12.4e} "
          f"{blk.mean():>12.4e} {blk.std():>12.4e}")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — CLASS SEPARABILITY (Cohen's d — Insomnia vs Normal)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 5: Top Discriminative Features — Cohen's d (Insomnia vs Normal)")
print(f"{'═'*60}")

X_ins    = X[y_group == 1]
X_nor    = X[y_group == 0]
mean_ins = X_ins.mean(axis=0)
mean_nor = X_nor.mean(axis=0)
std_pool = np.sqrt(
    (np.var(X_ins, axis=0) * (len(X_ins) - 1) +
     np.var(X_nor, axis=0) * (len(X_nor) - 1)) /
    (len(X_ins) + len(X_nor) - 2 + 1e-12)
)
cohens_d = (mean_ins - mean_nor) / (std_pool + 1e-12)

top15_idx = np.argsort(np.abs(cohens_d))[::-1][:15]
print(f"{'Rank':<5} {'Feature':<42} {'Cohen d':>9} {'Mean_Ins':>12} {'Mean_Nor':>12}")
print(f"{'─'*82}")
for rank, idx in enumerate(top15_idx):
    print(f"  {rank+1:<4} {feature_names[idx]:<42} "
          f"{cohens_d[idx]:>9.3f} "
          f"{mean_ins[idx]:>12.4e} "
          f"{mean_nor[idx]:>12.4e}")

# Per-block mean |Cohen's d| summary
print(f"\nMean |Cohen's d| per feature block:")
for block_name, (lo, hi) in FEATURE_BLOCKS.items():
    mean_d = np.abs(cohens_d[lo:hi]).mean()
    max_d  = np.abs(cohens_d[lo:hi]).max()
    print(f"  {block_name:<22}: mean|d|={mean_d:.3f}  max|d|={max_d:.3f}")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — SUBJECT-LEVEL OUTLIER DETECTION
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 6: Subject-Level Outlier Detection (L2 distance from centroid)")
print(f"{'═'*60}")

subj_means = np.zeros((len(subject_registry), D))
for sid in np.unique(subject_ids):
    subj_means[sid] = X[subject_ids == sid].mean(axis=0)

sm_std   = (subj_means - subj_means.mean(axis=0)) / (subj_means.std(axis=0) + 1e-12)
centroid = sm_std.mean(axis=0)
distances = np.linalg.norm(sm_std - centroid, axis=1)
z_dist    = (distances - distances.mean()) / (distances.std() + 1e-12)

print(f"{'Subject':<22} {'Epochs':>7} {'Dist':>16} {'Z-score':>9}  Flag")
print(f"{'─'*65}")
for sid in np.unique(subject_ids):
    name = id_to_name[sid]
    n_ep = (subject_ids == sid).sum()
    flag = " ← OUTLIER" if abs(z_dist[sid]) > 2.5 else \
           (" ← WARNING" if abs(z_dist[sid]) > 1.5 else "")
    print(f"  {name:<20} {n_ep:>7}  {distances[sid]:>14.3f}  {z_dist[sid]:>9.2f}{flag}")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — NORMALIZATION RECOMMENDATION
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 7: Normalization Recommendation (data-driven)")
print(f"{'═'*60}")

bp_abs_skew = np.abs(skew_arr[0:36]).mean()
bp_rel_skew = np.abs(skew_arr[36:72]).mean()
pac_skew    = np.abs(skew_arr[72:78]).mean()
wsmi_skew   = np.abs(skew_arr[78:108]).mean()
hjorth_skew = np.abs(skew_arr[108:126]).mean()
lz_skew     = np.abs(skew_arr[126:132]).mean()
sha_skew    = np.abs(skew_arr[132:240]).mean()

def _rec(skew: float, bounded: bool = False) -> str:
    if bounded:
        return "→ as-is  (bounded [0,1])"
    return "→ log1p THEN RobustScaler" if skew > 1.5 else "→ RobustScaler"

print(f"""
Feature Block          Mean|Skew|  Recommendation
──────────────────────────────────────────────────────────────────────
Band Power (abs)        {bp_abs_skew:>5.2f}    {_rec(bp_abs_skew)}
Band Power (rel)        {bp_rel_skew:>5.2f}    {_rec(bp_rel_skew, bounded=True)}
PAC MVL                 {pac_skew:>5.2f}    {_rec(pac_skew)}
wSMI                    {wsmi_skew:>5.2f}    {_rec(wsmi_skew)}
Hjorth                  {hjorth_skew:>5.2f}    {_rec(hjorth_skew)}
Lempel-Ziv              {lz_skew:>5.2f}    {_rec(lz_skew, bounded=True)}
Sharma Wavelet (all)    {sha_skew:>5.2f}    {_rec(sha_skew)}
  L1  sub-block              {"→ log1p mandatory (sum of abs values, power-law scale)"}
  L2  sub-block              {"→ log1p mandatory (RMS scale, same distribution)"}
  Linf sub-block             {"→ log1p mandatory (peak amplitude — extreme outlier risk)"}

CRITICAL RULES:
  1. ALL log1p transforms are applied to strictly non-negative features only.
     Wavelet norms (L1, L2, Linf) are always ≥ 0 — safe.
     Band power abs and PAC MVL are always ≥ 0 — safe.
     Hjorth Activity (variance) is always ≥ 0 — safe.
     Hjorth Mobility/Complexity — verify min ≥ 0 before applying log1p.

  2. Use RobustScaler (median/IQR) over StandardScaler because:
       a) Non-Gaussian features even after log transform.
       b) Single-epoch artifact subjects (e.g. 1-epoch N3 files) create extreme outliers.
       c) Linf norms are peak-amplitude — disproportionately sensitive to artifacts.

  3. DO NOT use MinMaxScaler — one artifact epoch collapses the entire range.

  4. Fit scaler on TRAIN fold only. Transform test fold with train scaler.
     Violating this is data leakage and inflates accuracy by 2-5%.

IMPLEMENTATION TEMPLATE (inside LOSO loop):
    import numpy as np
    from sklearn.preprocessing import RobustScaler

    # Indices that need log1p (non-negative, right-skewed)
    LOG1P_INDICES = list(range(0, 36))      # Band power abs
    LOG1P_INDICES += list(range(72, 78))    # PAC MVL
    LOG1P_INDICES += list(range(108, 114))  # Hjorth Activity only (every 3rd: act, mob, comp)
    LOG1P_INDICES += list(range(132, 240))  # All Sharma wavelet norms

    X_train_t = X_train.copy()
    X_test_t  = X_test.copy()
    X_train_t[:, LOG1P_INDICES] = np.log1p(X_train_t[:, LOG1P_INDICES])
    X_test_t[:,  LOG1P_INDICES] = np.log1p(X_test_t[:,  LOG1P_INDICES])

    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_t)
    X_test_scaled  = scaler.transform(X_test_t)
""")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 8 — PLOTS
# ══════════════════════════════════════════════════════════════════════════════
if HAS_PLOT:
    print(f"\n{'═'*60}")
    print("SECTION 8: Generating Plots → saving to", OUTPUT_DIR)
    print(f"{'═'*60}")

    # ── Plot 1: Skewness distribution per feature block (7 blocks) ────────────
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes = axes.flatten()
    for ax, (block_name, (lo, hi)) in zip(axes, FEATURE_BLOCKS.items()):
        ax.hist(skew_arr[lo:hi], bins=20, edgecolor='black', color='steelblue', alpha=0.8)
        ax.axvline(0,  color='green',  lw=1.5, linestyle='--', label='skew=0')
        ax.axvline(-1, color='orange', lw=1.0, linestyle=':')
        ax.axvline(+1, color='orange', lw=1.0, linestyle=':',  label='|skew|=1')
        ax.axvline(-2, color='red',    lw=1.0, linestyle=':')
        ax.axvline(+2, color='red',    lw=1.0, linestyle=':',  label='|skew|=2')
        ax.set_title(f"{block_name}\n(idx {lo}–{hi}, n={hi-lo})")
        ax.set_xlabel("Skewness")
        ax.set_ylabel("Count")
        ax.legend(fontsize=6)
    # hide the unused 8th subplot
    if len(FEATURE_BLOCKS) < 8:
        axes[7].axis('off')
    plt.suptitle("Skewness Distribution per Feature Block (Combined 240 Features)\n"
                 "|skew|>1 → log transform; |skew|>2 → mandatory", fontsize=12, y=1.01)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/plot1_skewness_per_block.png", dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("  Saved: plot1_skewness_per_block.png")

    # ── Plot 2: Cohen's d bar chart (top 20) ──────────────────────────────────
    top20_idx = np.argsort(np.abs(cohens_d))[::-1][:20]
    top20_d   = cohens_d[top20_idx]
    top20_nm  = [feature_names[i] for i in top20_idx]
    colors    = ['tomato' if d > 0 else 'steelblue' for d in top20_d]

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(range(20), top20_d, color=colors, edgecolor='black', alpha=0.85)
    ax.axhline(0,    color='black',  lw=0.8)
    ax.axhline(+0.5, color='orange', lw=1.0, linestyle='--', label="medium (|d|=0.5)")
    ax.axhline(-0.5, color='orange', lw=1.0, linestyle='--')
    ax.axhline(+0.8, color='red',    lw=1.0, linestyle='--', label="large (|d|=0.8)")
    ax.axhline(-0.8, color='red',    lw=1.0, linestyle='--')
    ax.set_xticks(range(20))
    ax.set_xticklabels(top20_nm, fontsize=7, rotation=45, ha='right')
    ax.set_ylabel("Cohen's d  (Insomnia − Normal) / SD_pooled")
    ax.set_title("Top 20 Features by Effect Size: Insomnia vs Normal\n"
                 "Red = Insomnia higher | Blue = Normal higher")
    ax.legend()
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/plot2_cohens_d_top20.png", dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("  Saved: plot2_cohens_d_top20.png")

    # ── Plot 3: Cohen's d mean per block (both pipelines side by side) ────────
    block_names_list = list(FEATURE_BLOCKS.keys())
    mean_d_per_block = [np.abs(cohens_d[lo:hi]).mean()
                        for lo, hi in FEATURE_BLOCKS.values()]
    max_d_per_block  = [np.abs(cohens_d[lo:hi]).max()
                        for lo, hi in FEATURE_BLOCKS.values()]
    x = np.arange(len(block_names_list))

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - 0.2, mean_d_per_block, width=0.35, label="Mean |d|",
           color='steelblue', edgecolor='black', alpha=0.85)
    ax.bar(x + 0.2, max_d_per_block,  width=0.35, label="Max |d|",
           color='tomato',   edgecolor='black', alpha=0.85)
    ax.axhline(0.5, color='orange', lw=1.0, linestyle='--', label="medium (0.5)")
    ax.axhline(0.8, color='red',    lw=1.0, linestyle='--', label="large (0.8)")
    ax.set_xticks(x)
    ax.set_xticklabels(block_names_list, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel("|Cohen's d|")
    ax.set_title("Discriminative Power per Feature Block (Insomnia vs Normal)")
    ax.legend()
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/plot3_cohens_d_per_block.png", dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("  Saved: plot3_cohens_d_per_block.png")

    # ── Plot 4: Per-stage relative band power — Insomnia vs Normal ────────────
    band_names = ["delta", "theta", "alpha", "sigma", "beta", "gamma"]
    fig, axes  = plt.subplots(1, 5, figsize=(20, 5), sharey=False)

    for ax, stage_id in zip(axes, range(5)):
        mask_ins = (y_group == 1) & (y_stage == stage_id)
        mask_nor = (y_group == 0) & (y_stage == stage_id)

        if mask_ins.sum() < 5 or mask_nor.sum() < 5:
            ax.set_title(f"{STAGE_NAMES[stage_id]}\n(insufficient data)")
            continue

        # Relative band power block: [36:72], shape after reshape = (6ch, 6bands)
        rel_ins = X[mask_ins, 36:72].mean(axis=0).reshape(6, 6)
        rel_nor = X[mask_nor, 36:72].mean(axis=0).reshape(6, 6)
        # Average across frontocentral: C4A1=0, C3A2=1, F3A2=2, F4A1=3
        mean_ins_band = rel_ins[[0, 1, 2, 3], :].mean(axis=0)
        mean_nor_band = rel_nor[[0, 1, 2, 3], :].mean(axis=0)

        xb = np.arange(6)
        ax.bar(xb - 0.2, mean_ins_band, width=0.35, label="Insomnia",
               color='tomato', alpha=0.85, edgecolor='black')
        ax.bar(xb + 0.2, mean_nor_band, width=0.35, label="Normal",
               color='steelblue', alpha=0.85, edgecolor='black')
        ax.set_xticks(xb)
        ax.set_xticklabels(band_names, fontsize=8)
        ax.set_title(f"{STAGE_NAMES[stage_id]}\nIns={mask_ins.sum()} Nor={mask_nor.sum()}")
        ax.set_ylabel("Relative Power" if stage_id == 0 else "")
        if stage_id == 0:
            ax.legend(fontsize=8)

    plt.suptitle("Mean Relative Band Power per Stage: Insomnia vs Normal\n"
                 "(Frontocentral channels — C4A1, C3A2, F3A2, F4A1 averaged)",
                 fontsize=11, y=1.02)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/plot4_band_power_per_stage.png", dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("  Saved: plot4_band_power_per_stage.png")

    # ── Plot 5: Sharma norm-type skewness comparison ───────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (norm_name, indices) in zip(axes, SHARMA_NORM_BLOCKS.items()):
        ax.hist(skew_arr[indices], bins=15, edgecolor='black', color='mediumpurple', alpha=0.8)
        ax.axvline(0,    color='green',  lw=1.5, linestyle='--')
        ax.axvline(+1.5, color='orange', lw=1.0, linestyle=':')
        ax.axvline(-1.5, color='orange', lw=1.0, linestyle=':')
        ax.set_title(f"Sharma — {norm_name}\n(n={len(indices)} features)")
        ax.set_xlabel("Skewness")
        ax.set_ylabel("Count")
    plt.suptitle("Sharma Wavelet Norm Skewness by Norm Type\n"
                 "Orange dashed = |skew|=1.5 threshold", y=1.05)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/plot5_sharma_norm_skewness.png", dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("  Saved: plot5_sharma_norm_skewness.png")

    # ── Plot 6: Subject outlier distances ──────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 5))
    subj_names  = [id_to_name[sid] for sid in range(len(subject_registry))]
    bar_colors  = ['tomato'    if abs(z_dist[sid]) > 2.5 else
                  ('gold'      if abs(z_dist[sid]) > 1.5 else 'steelblue')
                   for sid in range(len(subject_registry))]
    ax.bar(range(len(subject_registry)), distances,
           color=bar_colors, edgecolor='black', alpha=0.85)
    ax.set_xticks(range(len(subject_registry)))
    ax.set_xticklabels(subj_names, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel("L2 Distance from Group Centroid (standardized features)")
    ax.set_title("Subject-Level Outlier Detection\nRed = z > 2.5 | Yellow = z > 1.5")
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/plot6_subject_outliers.png", dpi=150, bbox_inches='tight')
    plt.close(fig)
    print("  Saved: plot6_subject_outliers.png")

    print(f"\nAll plots saved to {OUTPUT_DIR}/")

else:
    print("Plots skipped — matplotlib not available.")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 9 — FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 9: Inspection Complete")
print(f"{'═'*60}")
print(f"  Dataset      : {ML_DIR}")
print(f"  Total epochs : {N}")
print(f"  Total features: {D}  (132 classic + 108 Sharma)")
print(f"  Subjects     : {len(np.unique(subject_ids))}")
print(f"  Output plots : {OUTPUT_DIR}")
print(f"\nNext step: train_loso.py — apply normalization template from Section 7.")

Loading combined dataset...
X shape         : (13007, 240)   (N=13007 epochs, D=240 features)
Unique subjects : 22
Unique stages   : [0 1 2 3 4]
Unique groups   : [0 1]

Feature layout confirmed:
  [  0: 36]  Band Power (abs)  (36 features)
  [ 36: 72]  Band Power (rel)  (36 features)
  [ 72: 78]  PAC MVL  (6 features)
  [ 78:108]  wSMI  (30 features)
  [108:126]  Hjorth  (18 features)
  [126:132]  Lempel-Ziv  (6 features)
  [132:240]  Sharma Wavelet  (108 features)

════════════════════════════════════════════════════════════
SECTION 2: Pathological Value Audit
════════════════════════════════════════════════════════════
Features with any NaN         : 0  (must be 0)
Features with any Inf         : 0  (must be 0)
Features with zero variance   : 54  (useless for ML)
  Zero-variance features: ['bp_abs_delta_ch0', 'bp_abs_theta_ch0', 'bp_abs_alpha_ch0', 'bp_abs_sigma_ch0', 'bp_abs_beta_ch0', 'bp_abs_gamma_ch0', 'bp_abs_delta_ch1', 'bp_abs_theta_ch1', 'bp_abs_alpha_ch1', 'bp_abs_sigma_ch1

In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# 31-COMBINATION POOLED 10-FOLD CV — COMBINED FEATURES (240-D)
# Sharma-style replication upper bound (NOT subject-independent)
# ═══════════════════════════════════════════════════════════════════════════════

from __future__ import annotations

# ── 0. DEPENDENCIES ────────────────────────────────────────────────────────────
import subprocess
import sys

for pkg in ["xgboost", "scikit-learn", "numpy", "scipy", "pandas"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# ── 1. IMPORTS ─────────────────────────────────────────────────────────────────
import json
import time
import warnings
import itertools
from pathlib import Path
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd

from sklearn.base import clone as sk_clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
ML_DIR = "/kaggle/working/ml_ready_combined"
OUTPUT_DIR = "/kaggle/working/four_feature"
SEED = 42
N_FOLDS = 10
SKIP_SVM = True
USE_AUDIT_TRANSFORM = True

np.random.seed(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

STAGE_MAP = {0: "W", 1: "N1", 2: "N2", 3: "N3", 4: "R"}
SHARMA_ALIASES = {(1, 2): "LSS", (1, 2, 3): "NREM", (0, 1, 2, 3, 4): "ALL"}

# ── 240-D FEATURE LAYOUT ──────────────────────────────────────────────────────
# [  0: 36] Band Power abs
# [ 36: 72] Band Power rel
# [ 72: 78] PAC MVL
# [ 78:108] wSMI
# [108:126] Hjorth (act,mob,comp per ch)
# [126:132] Lempel-Ziv
# [132:240] Sharma Wavelet (108)

def get_log1p_indices() -> np.ndarray:
    """
    Conservative audit-driven transform policy:
      log1p -> bp_abs, pac, wsmi, hjorth_activity_only, wavelets
      as-is -> bp_rel, hjorth_mob, hjorth_comp, lz
    """
    idx = []
    idx.extend(range(0, 36))       # bp_abs
    idx.extend(range(72, 78))      # pac
    idx.extend(range(78, 108))     # wsmi
    idx.extend([108, 111, 114, 117, 120, 123])  # hjorth act only
    idx.extend(range(132, 240))    # wavelets
    return np.array(sorted(set(idx)), dtype=np.int32)

LOG1P_INDICES = get_log1p_indices()

def apply_audit_transform(X: np.ndarray) -> np.ndarray:
    X_t = X.astype(np.float64, copy=True)
    if X_t.shape[1] != 240:
        raise ValueError(f"Expected 240 features, got {X_t.shape[1]}")

    # hard check before log (don't silently hide negatives)
    mins = X_t[:, LOG1P_INDICES].min(axis=0)
    if np.any(mins < -1e-12):
        raise ValueError("Negative values found in log1p-designated features. Fix extractor or index map.")

    X_t[:, LOG1P_INDICES] = np.log1p(np.clip(X_t[:, LOG1P_INDICES], 0.0, None))

    if np.isnan(X_t).any() or np.isinf(X_t).any():
        raise ValueError("NaN/Inf after audit transform")
    return X_t

# ══════════════════════════════════════════════════════════════════════════════
# SECTION A — LOAD & PREPROCESS
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("SECTION A — Loading and Preprocessing")
print("=" * 70)

p = Path(ML_DIR)
X_raw = np.load(p / "X.npy")
y_stage = np.load(p / "y_stage.npy")
y_group = np.load(p / "y_group.npy")
subject_ids = np.load(p / "subject_ids.npy")

with open(p / "metadata.json", "r") as f:
    meta = json.load(f)

feature_names = np.array(meta["feature_names"])

print(f"  X_raw       : {X_raw.shape}")
print(f"  y_stage     : {np.unique(y_stage, return_counts=True)}")
print(f"  y_group     : {np.unique(y_group, return_counts=True)}")
print(f"  Subjects    : {len(np.unique(subject_ids))}")

assert X_raw.ndim == 2
assert y_stage.ndim == y_group.ndim == subject_ids.ndim == 1
assert X_raw.shape[0] == y_stage.shape[0] == y_group.shape[0] == subject_ids.shape[0]
assert int(meta["n_features"]) == X_raw.shape[1]
assert len(feature_names) == X_raw.shape[1]
assert X_raw.shape[1] == 240, f"Expected 240 features, got {X_raw.shape[1]}"

if USE_AUDIT_TRANSFORM:
    X_t = apply_audit_transform(X_raw)
    print("  Transform policy   : AUDIT-driven blockwise log1p")
else:
    X_t = np.log1p(np.clip(X_raw.astype(np.float64), 0.0, None))
    print("  Transform policy   : Global log1p (fallback)")

assert not np.isnan(X_t).any() and not np.isinf(X_t).any()

vt = VarianceThreshold(threshold=1e-6)
X_filtered = vt.fit_transform(X_t).astype(np.float32)
support = vt.get_support()
kept_names = feature_names[support]
dropped_names = feature_names[~support]

print(f"  Input features     : {X_raw.shape[1]}")
print(f"  Zero-var dropped   : {(~support).sum()}")
print(f"  Remaining features : {X_filtered.shape[1]}")

out = Path(OUTPUT_DIR)
np.save(out / "kept_feature_names.npy", kept_names)
np.save(out / "dropped_feature_names.npy", dropped_names)

with open(out / "preprocess_manifest.json", "w") as f:
    json.dump({
        "ml_dir": ML_DIR,
        "use_audit_transform": USE_AUDIT_TRANSFORM,
        "log1p_indices": LOG1P_INDICES.tolist(),
        "input_features": int(X_raw.shape[1]),
        "dropped_features": int((~support).sum()),
        "remaining_features": int(X_filtered.shape[1]),
        "dropped_feature_names": dropped_names.tolist(),
    }, f, indent=2)

# ══════════════════════════════════════════════════════════════════════════════
# SECTION B — COMBINATIONS
# ══════════════════════════════════════════════════════════════════════════════
all_combos: List[Tuple[int, ...]] = []
for r in range(1, 6):
    all_combos.extend(itertools.combinations([0, 1, 2, 3, 4], r))
assert len(all_combos) == 31

def combo_label(combo: Tuple[int, ...]) -> str:
    alias = SHARMA_ALIASES.get(combo)
    s = "+".join(STAGE_MAP[k] for k in combo)
    return f"{s} ({alias})" if alias else s

def combo_epoch_counts(combo: Tuple[int, ...]) -> Dict[str, int]:
    m = np.isin(y_stage, list(combo))
    ys = y_group[m]
    ins = int((ys == 1).sum())
    nor = int((ys == 0).sum())
    return {"total": ins + nor, "insomnia": ins, "normal": nor}

print(f"\nSECTION B — {len(all_combos)} stage combinations")
for c in all_combos:
    cc = combo_epoch_counts(c)
    print(f"  {combo_label(c):<32} total={cc['total']:>6} Ins={cc['insomnia']:>5} Nor={cc['normal']:>5}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION C — MODELS / METRICS
# ══════════════════════════════════════════════════════════════════════════════
def get_models(seed: int = SEED, skip_svm: bool = SKIP_SVM) -> Dict[str, object]:
    models: Dict[str, object] = {
        "Dummy": DummyClassifier(strategy="most_frequent", random_state=seed),
        "LogReg": LogisticRegression(C=1.0, max_iter=3000, solver="saga",
                                     class_weight="balanced", n_jobs=-1, random_state=seed),
        "KNN_k7": KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
        "DecisionTree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=10,
                                               class_weight="balanced", random_state=seed),
        "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                                               class_weight="balanced", n_jobs=-1, random_state=seed),
        "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                 subsample=0.8, colsample_bytree=0.8,
                                 objective="binary:logistic", eval_metric="logloss",
                                 n_jobs=-1, random_state=seed, verbosity=0),
    }
    if not skip_svm:
        models["SVM_RBF"] = SVC(C=1.0, kernel="rbf", gamma="scale",
                                class_weight="balanced", probability=True, random_state=seed)
    else:
        print("  [INFO] SVM skipped (SKIP_SVM=True)")
    return models

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray | None) -> Dict[str, float]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    auc = 0.5
    if y_prob is not None and len(np.unique(y_true)) > 1:
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = 0.5
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "specificity": float(spec),
        "sensitivity": float(sens),
        "f1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0),
        "auroc": auc,
    }

def run_10fold_cv(X: np.ndarray, y: np.ndarray, model_name: str, model: object, seed: int = SEED) -> Dict[str, float]:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    folds = []

    for i, (tr, te) in enumerate(skf.split(X, y)):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        scaler = RobustScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        clf = sk_clone(model)
        try:
            clf.fit(Xtr, ytr)
        except Exception as e:
            print(f"      [WARN] {model_name} fold {i} fit failed: {e}")
            continue

        yp = clf.predict(Xte)
        yprob = None
        if hasattr(clf, "predict_proba"):
            try:
                yprob = clf.predict_proba(Xte)[:, 1]
            except Exception:
                pass
        elif hasattr(clf, "decision_function"):
            try:
                yprob = clf.decision_function(Xte)
            except Exception:
                pass

        folds.append(compute_metrics(yte, yp, yprob))

    if not folds:
        return {}

    outm: Dict[str, float] = {}
    for k in folds[0]:
        vals = [f[k] for f in folds]
        outm[f"{k}_mean"] = float(np.mean(vals))
        outm[f"{k}_std"] = float(np.std(vals))
    return outm

# ══════════════════════════════════════════════════════════════════════════════
# SECTION D — RUN
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("SECTION D — Running 31 × N_classifiers 10-fold CV")
print("  ⚠ pooled epoch CV only (leakage by design)")
print("=" * 70)

models = get_models()
model_names = list(models.keys())
master_results: Dict[str, Dict[str, Dict[str, float] | Dict[str, int]]] = {}

total_runs = len(all_combos) * len(model_names)
run_count = 0
t_global = time.time()

for combo in all_combos:
    label = combo_label(combo)
    mask = np.isin(y_stage, list(combo))
    X_sub = X_filtered[mask]
    y_sub = y_group[mask]
    c = combo_epoch_counts(combo)

    if c["insomnia"] < N_FOLDS or c["normal"] < N_FOLDS:
        print(f"\n  [SKIP] {label} — insufficient class counts (Ins={c['insomnia']}, Nor={c['normal']})")
        continue

    print(f"\n  {'─'*68}")
    print(f"  COMBO: {label:<32} N={c['total']} Ins={c['insomnia']} Nor={c['normal']}")
    print(f"  {'─'*68}")

    master_results[label] = {"_meta": c}

    for mn, model in models.items():
        run_count += 1
        t0 = time.time()
        r = run_10fold_cv(X_sub, y_sub, mn, model)
        dt = time.time() - t0
        master_results[label][mn] = r

        if r:
            print(f"    {mn:<14} acc={r['accuracy_mean']:.4f}±{r['accuracy_std']:.3f} "
                  f"sens={r['sensitivity_mean']:.4f} spec={r['specificity_mean']:.4f} "
                  f"f1={r['f1_mean']:.4f} kappa={r['kappa_mean']:.4f} auc={r['auroc_mean']:.4f} "
                  f"[{dt:.1f}s] ({run_count}/{total_runs})")
        else:
            print(f"    {mn:<14} [FAILED]")

print(f"\n  Total runtime: {(time.time() - t_global)/60:.1f} min")
print(f"  Completed combos   : {len(master_results)} / 31")
print(f"  Models evaluated   : {len(model_names)}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION E — TABLES + SAVE
# ═══════════════════════════��══════════════════════════════════════════════════
METRIC_COLS = ["accuracy", "precision", "specificity", "sensitivity", "f1", "kappa", "auroc"]

rows = []
for label, dd in master_results.items():
    meta_c = dd.get("_meta", {})
    for mn in model_names:
        res = dd.get(mn, {})
        if not isinstance(res, dict) or not res:
            continue
        row = {"Stage Combo": label, "N Epochs": meta_c.get("total", "?"), "Model": mn}
        for m in METRIC_COLS:
            cap = m.capitalize()
            row[cap] = res.get(f"{m}_mean", np.nan)
            row[f"{cap}_std"] = res.get(f"{m}_std", np.nan)
        rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(out / "sharma_31combo_all_results.csv", index=False)

for mn in model_names:
    sub = df[df["Model"] == mn]
    if not sub.empty:
        sub.to_csv(out / f"table7_{mn}.csv", index=False)

def _jsonify(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

with open(out / "sharma_31combo_master.json", "w") as f:
    json.dump(
        {k: {m: {mk: _jsonify(mv) for mk, mv in mv2.items()}
              for m, mv2 in v.items() if isinstance(mv2, dict)}
         for k, v in master_results.items()},
        f, indent=2
    )

best_rows = []
for mn in model_names:
    sub = df[df["Model"] == mn]
    if not sub.empty and not sub["Kappa"].isna().all():
        best_rows.append(sub.loc[sub["Kappa"].idxmax()].to_dict())
pd.DataFrame(best_rows).to_csv(out / "best_per_model_summary.csv", index=False)

print(f"\nSaved to: {OUTPUT_DIR}")
print("  - sharma_31combo_all_results.csv")
print("  - table7_<model>.csv")
print("  - sharma_31combo_master.json")
print("  - best_per_model_summary.csv")
print("  - kept_feature_names.npy")
print("  - dropped_feature_names.npy")
print("  - preprocess_manifest.json")

print(f"""
{'═'*70}
INTERPRETATION GUARD
{'═'*70}
⚠ Pooled 10-fold CV on epochs leaks subject identity across folds.
⚠ Use LOSO/session-wise CV for real generalization claims.
{'═'*70}
""")

SECTION A — Loading and Preprocessing
  X_raw       : (13007, 240)
  y_stage     : (array([0, 1, 2, 3, 4], dtype=int32), array([3098, 4605, 2272, 2052,  980]))
  y_group     : (array([0, 1], dtype=int32), array([6705, 6302]))
  Subjects    : 22
  Transform policy   : AUDIT-driven blockwise log1p
  Input features     : 240
  Zero-var dropped   : 132
  Remaining features : 108

SECTION B — 31 stage combinations
  W                                total=  3098 Ins= 2165 Nor=  933
  N1                               total=  4605 Ins= 2065 Nor= 2540
  N2                               total=  2272 Ins=  736 Nor= 1536
  N3                               total=  2052 Ins=  759 Nor= 1293
  R                                total=   980 Ins=  577 Nor=  403
  W+N1                             total=  7703 Ins= 4230 Nor= 3473
  W+N2                             total=  5370 Ins= 2901 Nor= 2469
  W+N3                             total=  5150 Ins= 2924 Nor= 2226
  W+R                              total=  